In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path


# ============================================================
# User settings
# ============================================================

ROOT = Path.cwd()

DATA_ROOT = ROOT / "Data" / "jacobian_data"
OUTPUT_ROOT = ROOT / "Results" / "jacobian_sensitivity"
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

NODES = [4, 6, 8]

# Tick spacing per node
NODE_PLOT_CONFIGS = {
    4: {"x_tick_step": 10, "y_tick_step": 8},
    6: {"x_tick_step": 10, "y_tick_step": 16},
    8: {"x_tick_step": 20, "y_tick_step": 32},
}

# Font sizes
LABEL_FONTSIZE = 46
TICK_FONTSIZE = 32
CBAR_LABEL_FONTSIZE = 42
CBAR_TICK_FONTSIZE = 28
PANEL_FONTSIZE = 38

DPI = 600

plt.rcParams.update({
    "font.family": "serif",
    "font.size": 28,
    "axes.labelsize": LABEL_FONTSIZE,
    "xtick.labelsize": TICK_FONTSIZE,
    "ytick.labelsize": TICK_FONTSIZE,
    "mathtext.fontset": "dejavuserif",
    "axes.linewidth": 1.8,
})


# ============================================================
# Helper functions
# ============================================================

def load_jacobian_matrices(results_dir: Path):
    """Load NN, PyTheus, and difference sensitivity matrices."""
    required_files = {
        "mean_nn": results_dir / "mean_nn_sensitivity.npy",
        "mean_py": results_dir / "mean_pytheus_sensitivity.npy",
        "difference": results_dir / "difference_sensitivity.npy",
    }

    for name, path in required_files.items():
        if not path.exists():
            raise FileNotFoundError(f"Missing {name} file: {path}")

    mean_nn = np.load(required_files["mean_nn"])
    mean_py = np.load(required_files["mean_py"])
    difference = np.load(required_files["difference"])

    return mean_nn, mean_py, difference


def style_axis(
    ax,
    matrix,
    x_tick_step,
    y_tick_step,
    show_ylabel=False,
    show_yticklabels=True,
    show_yticks=True,
):
    x_ticks = np.arange(0, matrix.shape[1], x_tick_step)
    y_ticks = np.arange(0, matrix.shape[0], y_tick_step)

    ax.set_xticks(x_ticks)
    ax.set_yticks(y_ticks)

    ax.set_xticklabels(x_ticks, rotation=0)

    if show_yticklabels:
        ax.set_yticklabels(y_ticks)
    else:
        ax.set_yticklabels([])

    ax.tick_params(
        axis="x",
        which="major",
        length=8,
        width=1.6,
        direction="out",
    )

    ax.tick_params(
        axis="y",
        which="major",
        left=show_yticks,
        right=False,
        length=8 if show_yticks else 0,
        width=1.6,
        direction="out",
        labelleft=show_yticklabels,
    )

    if show_ylabel:
        ax.set_ylabel("Output amplitude index", labelpad=18)
    else:
        ax.set_ylabel("")

    ax.set_xlabel("")
    ax.set_box_aspect(1)


def add_panel_label(ax, label):
    ax.text(
        -0.13,
        1.03,
        label,
        transform=ax.transAxes,
        fontsize=PANEL_FONTSIZE,
        fontweight="bold",
        va="bottom",
        ha="left",
    )


def plot_jacobian_sensitivity_for_node(node: int):
    """Create and save the 3-panel Jacobian sensitivity figure for one node."""
    results_dir = DATA_ROOT / f"{node}_node"
    output_dir = OUTPUT_ROOT / f"{node}_node"
    output_dir.mkdir(parents=True, exist_ok=True)

    x_tick_step = NODE_PLOT_CONFIGS[node]["x_tick_step"]
    y_tick_step = NODE_PLOT_CONFIGS[node]["y_tick_step"]

    # ============================================================
    # Load matrices
    # ============================================================

    mean_nn, mean_py, difference = load_jacobian_matrices(results_dir)

    print("\n" + "=" * 80)
    print(f"Node {node}")
    print("=" * 80)
    print("Input directory:", results_dir)
    print("mean_nn shape:", mean_nn.shape)
    print("mean_py shape:", mean_py.shape)
    print("difference shape:", difference.shape)

    common_vmin = 0.0
    common_vmax = max(np.max(mean_nn), np.max(mean_py))

    # ============================================================
    # Horizontal figure layout
    # panel a | panel b | shared cbar | small gap | panel c | cbar c
    # ============================================================

    fig = plt.figure(figsize=(38, 12), constrained_layout=False)

    gs = fig.add_gridspec(
        1,
        6,
        width_ratios=[1.0, 1.0, 0.055, 0.045, 1.0, 0.055],
        wspace=0.08,
    )

    ax_a = fig.add_subplot(gs[0, 0])
    ax_b = fig.add_subplot(gs[0, 1])
    cax_ab = fig.add_subplot(gs[0, 2])

    ax_gap = fig.add_subplot(gs[0, 3])
    ax_gap.axis("off")

    ax_c = fig.add_subplot(gs[0, 4])
    cax_c = fig.add_subplot(gs[0, 5])

    # ============================================================
    # Panel (a): NN sensitivity
    # ============================================================

    im_a = ax_a.imshow(
        mean_nn,
        origin="lower",
        interpolation="nearest",
        cmap="viridis",
        vmin=common_vmin,
        vmax=common_vmax,
        aspect="auto",
    )

    style_axis(
        ax_a,
        mean_nn,
        x_tick_step=x_tick_step,
        y_tick_step=y_tick_step,
        show_ylabel=True,
        show_yticklabels=True,
        show_yticks=True,
    )

    add_panel_label(ax_a, "(a)")

    # ============================================================
    # Panel (b): PyTheus sensitivity
    # ============================================================

    im_b = ax_b.imshow(
        mean_py,
        origin="lower",
        interpolation="nearest",
        cmap="viridis",
        vmin=common_vmin,
        vmax=common_vmax,
        aspect="auto",
    )

    style_axis(
        ax_b,
        mean_py,
        x_tick_step=x_tick_step,
        y_tick_step=y_tick_step,
        show_ylabel=False,
        show_yticklabels=False,
        show_yticks=False,
    )

    add_panel_label(ax_b, "(b)")

    # Shared colorbar for panels (a) and (b)
    cbar_ab = fig.colorbar(im_b, cax=cax_ab)

    cbar_ab.set_label(
        "Mean Jacobian sensitivity",
        fontsize=CBAR_LABEL_FONTSIZE,
        labelpad=20,
    )

    cbar_ab.ax.yaxis.set_label_position("right")
    cbar_ab.ax.yaxis.set_ticks_position("right")

    cbar_ab.ax.tick_params(
        labelsize=CBAR_TICK_FONTSIZE,
        width=1.6,
        length=7,
    )

    # ============================================================
    # Panel (c): Difference
    # ============================================================

    im_c = ax_c.imshow(
        difference,
        origin="lower",
        interpolation="nearest",
        cmap="magma",
        vmin=0.0,
        vmax=np.max(difference),
        aspect="auto",
    )

    style_axis(
        ax_c,
        difference,
        x_tick_step=x_tick_step,
        y_tick_step=y_tick_step,
        show_ylabel=False,
        show_yticklabels=False,
        show_yticks=False,
    )

    add_panel_label(ax_c, "(c)")

    # Colorbar for panel (c)
    cbar_c = fig.colorbar(im_c, cax=cax_c)

    cbar_c.set_label(
        r"$|S^{\mathrm{NN}}_{ij} - S^{\mathrm{PM}}_{ij}|$",
        fontsize=CBAR_LABEL_FONTSIZE,
        labelpad=20,
    )

    cbar_c.ax.tick_params(
        labelsize=CBAR_TICK_FONTSIZE,
        width=1.6,
        length=7,
    )

    # ============================================================
    # Shared x-axis label
    # ============================================================

    fig.supxlabel(
        "Input graph-weight index",
        fontsize=LABEL_FONTSIZE,
        y=0.035,
    )

    fig.subplots_adjust(
        left=0.055,
        right=0.985,
        bottom=0.19,
        top=0.93,
        wspace=0.08,
    )

    # ============================================================
    # Save
    # ============================================================

    save_stem = output_dir / f"jacobian_sensitivity_horizontal_{node}_node"

    fig.savefig(save_stem.with_suffix(".png"), dpi=DPI, bbox_inches="tight")
    fig.savefig(save_stem.with_suffix(".pdf"), bbox_inches="tight")

    plt.close(fig)

    print("Saved:")
    print(save_stem.with_suffix(".png"))
    print(save_stem.with_suffix(".pdf"))


# ============================================================
# Run for 4, 6, and 8 nodes
# ============================================================

print("Current directory:", ROOT)
print("Data root:", DATA_ROOT)
print("Output root:", OUTPUT_ROOT)

for node in NODES:
    plot_jacobian_sensitivity_for_node(node)

Current directory: /home/bo48god/projects/deep_dreaming/PolySurrogate/plotting_code
Data root: /home/bo48god/projects/deep_dreaming/PolySurrogate/plotting_code/Data/jacobian_data
Output root: /home/bo48god/projects/deep_dreaming/PolySurrogate/plotting_code/Results/jacobian_sensitivity

Node 4
Input directory: /home/bo48god/projects/deep_dreaming/PolySurrogate/plotting_code/Data/jacobian_data/4_node
mean_nn shape: (16, 24)
mean_py shape: (16, 24)
difference shape: (16, 24)
Saved:
/home/bo48god/projects/deep_dreaming/PolySurrogate/plotting_code/Results/jacobian_sensitivity/4_node/jacobian_sensitivity_horizontal_4_node.png
/home/bo48god/projects/deep_dreaming/PolySurrogate/plotting_code/Results/jacobian_sensitivity/4_node/jacobian_sensitivity_horizontal_4_node.pdf

Node 6
Input directory: /home/bo48god/projects/deep_dreaming/PolySurrogate/plotting_code/Data/jacobian_data/6_node
mean_nn shape: (64, 60)
mean_py shape: (64, 60)
difference shape: (64, 60)
Saved:
/home/bo48god/projects/deep_dr